# NYC Taxi Trip Duration — compare regression models

Predict the length of a taxi ride in seconds. This is **regression**, not a future time-series forecast: train and test trips cover the same months. We use pickup information and destination coordinates, but never the actual drop-off time.


## Helper functions

These helper functions create model inputs and calculate Kaggle's score. The candidate models and shared-holdout comparison live in `model_comparison.py` beside this notebook. Defining helpers here does not train a model yet.


In [ ]:
def build_features(df):
    pickup = pd.to_datetime(df['pickup_datetime'])
    lat1 = np.radians(df['pickup_latitude'].astype(float))
    lat2 = np.radians(df['dropoff_latitude'].astype(float))
    dlat = lat2 - lat1
    dlon = np.radians(df['dropoff_longitude'].astype(float) - df['pickup_longitude'].astype(float))
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2

    features = pd.DataFrame(index=df.index)
    features['distance_km'] = 2 * 6371 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    features['pickup_hour'] = pickup.dt.hour
    features['pickup_weekday'] = pickup.dt.dayofweek
    features['pickup_month'] = pickup.dt.month
    features['pickup_latitude'] = df['pickup_latitude']
    features['pickup_longitude'] = df['pickup_longitude']
    features['dropoff_latitude'] = df['dropoff_latitude']
    features['dropoff_longitude'] = df['dropoff_longitude']
    features['passenger_count'] = df['passenger_count']
    features['vendor_id'] = df['vendor_id']
    features['store_and_fwd'] = (df['store_and_fwd_flag'] == 'Y').astype(int)
    return features


In [ ]:
def rmsle(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.clip(np.asarray(predicted), 0, None)
    return np.sqrt(np.mean((np.log1p(actual) - np.log1p(predicted)) ** 2))


## 1. Load and inspect

Each row is one trip. `trip_duration` is the target. `dropoff_datetime` is available only after the trip and would reveal the answer, so it is never passed to the model.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from model_comparison import compare_models, candidate_models

data_dir = Path('/kaggle/input/nyc-taxi-trip-duration')
if not data_dir.exists():
    data_dir = Path('data')

def data_file(stem):
    csv_file = data_dir / f'{stem}.csv'
    return csv_file if csv_file.exists() else data_dir / f'{stem}.zip'

train = pd.read_csv(data_file('train'), parse_dates=['pickup_datetime'])
test = pd.read_csv(data_file('test'), parse_dates=['pickup_datetime'])
print('Train:', train.shape, 'Test:', test.shape)
print('Missing train:', train.isna().sum().to_dict())
print('Train dates:', train.pickup_datetime.min(), 'to', train.pickup_datetime.max())
print('Test dates:', test.pickup_datetime.min(), 'to', test.pickup_datetime.max())
print('Trip duration seconds:', train.trip_duration.quantile([0, .5, .99, 1]).to_dict())


## 2. Build features and validation

We use straight-line distance, departure hour/day/month, location, passenger count, and vendor. The competition's train and test cover the same six months, so a random holdout is a reasonable first approximation of its hidden test. This is unlike Store Sales, where we predicted future dates and had to split past → future. The destination coordinates are competition-provided; a real app could use them only after the rider supplies a destination.


In [ ]:
X = build_features(train)
X_test = build_features(test)
y = train['trip_duration'].to_numpy()
assert X.columns.equals(X_test.columns)
assert 'dropoff_datetime' not in X and 'trip_duration' not in X
train_idx, valid_idx = train_test_split(np.arange(len(train)), test_size=0.20, random_state=42)
print('Features:', X.columns.tolist())
print('Train rows:', len(train_idx), 'Validation rows:', len(valid_idx))


## 3. Compare models on the same holdout

The baseline predicts one typical duration for every trip. We also try Histogram Gradient Boosting, Random Forest, and Extra Trees. All three train on the **same 150,000 trips** and are scored on the **same untouched validation trips**, so the comparison is fair. Lower RMSLE is better. Compare train versus validation too: a large gap warns about overfitting. The model definitions and scoring function are in `model_comparison.py` beside this notebook.


In [ ]:
scores, fitted_models = compare_models(X, y, train_idx, valid_idx, sample_size=150_000, random_state=42)
display(scores.round(5))
best_name = scores.loc[scores['model'] != 'Constant baseline'].nsmallest(1, 'valid_rmsle')['model'].iloc[0]
print('Best validation model:', best_name)


## 4. Predict test and create submission

In the second local run, Random Forest scored 0.42277 RMSLE on validation, ahead of Histogram Gradient Boosting (0.42925) and Extra Trees (0.44179) on the same 150,000-row training sample. Its train score (0.30396) is much lower, so overfitting remains a concern even though its validation score is best here. We retrain the selected model on 500,000 labeled trips, then check row count, IDs, missing values, and negative predictions before saving. A Kaggle late-submission score is practice feedback, not an original 2017 competition placement.


In [ ]:
final_idx = np.random.default_rng(42).choice(np.arange(len(train)), size=min(500_000, len(train)), replace=False)
final_model = candidate_models(42)[best_name]
final_model.fit(X.iloc[final_idx], np.log1p(y[final_idx]))
test_prediction = np.expm1(final_model.predict(X_test)).clip(min=0)
submission = pd.DataFrame({'id': test['id'], 'trip_duration': test_prediction})
assert len(submission) == len(test)
assert submission['id'].equals(test['id'])
assert np.isfinite(submission['trip_duration']).all()
assert submission['trip_duration'].ge(0).all()
output_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('output')
output_dir.mkdir(exist_ok=True)
submission_path = output_dir / f"submission_{best_name.lower().replace(' ', '_')}.csv"
submission.to_csv(submission_path, index=False)
print('Saved:', submission_path, 'Rows:', len(submission))
print(submission.head())
